In [1]:
import numpy as np

np.set_printoptions(precision=3,suppress=True)

In [2]:
import sys
from os.path import dirname

sys.path.append(dirname("../src/"))

In [3]:
from samosa.symmetry.representations import MatrixGroupElement, \
                                            PermutationGroupElement, \
                                            IdentityGroupElement

from samosa.symmetry.operations_3d import operator_C, operator_M, \
                                          operator_S

# Index

* [Brief description of the `GroupElement` object properties](#intro)
* [`MatrixGroupElement` - matrix representation of the group elements](#matrix)
    - [Common matrix operations (`operator_C`, `operator_M`, and `operator_S`)](#matrix-operations)
    - [Basis of the transformations](#matrix-basis)
    - [Initialization of the `MatrixGroupElement` object](#matrix-element-init)
    - [Product of two `MatrixGroupElement` objects](#matrix-element-prod)
    - [Inverse of a `MatrixGroupElement` object](#matrix-element-inv)
    - [Action of a `MatrixGroupElement` object on arrays](#matrix-element-act)
    - [`MatrixGroupElement` object for identity matrix](#matrix-element-id)
* [`PermutationGroupElement` - permutation representation of group elements](#permutation)
    - [Initialization of the `PermutationGroupElement` object](#permutation-element-init)
    - [Permutation of array elements and inegers](#permutation-element-act)
    - [Product of two `PermutationGroupElement` objects](#permutation-element-prod)
    - [Inverse of a `PermutationGroupElement` object](#permutation-element-inv)
    - [`PermutationGroupElement` object for trivial permutation](#permutation-element-id)
* [`IdentityGroupElement` - generic identity group element](#identity)

# Brief description of the `GroupElement` object properties <a name="intro"></a>

When analysing the symmetry properties of a given system, the interface between abstract group theory and symmetry transformations of physical objects is contained in how we represent the individual elements of the group. 
Two of the most common representations are __matrices__ and __permutations__.
`samosa` provides implementaions for these representations in the form of `MatrixGroupElement` and `PermutationGroupElement` classes contained in `samosa.symmetry.group_utils`. 
These two classes both inherit from the interface class `GroupElement` and therefore share the core properties and methods. 
In particular, `GroupElement` objects define

- Multiplication between elements in the same representation;
- Group action on other objects (arrays, numbers);
- Inverse, accessed through `.inv` property;
- Equality relation, which is used to determine if two group elements in the same representation correspond to the same element.

This notebook provides information about the definition and use of these objects.

# `MatrixGroupElement` - matrix representation of the group elements <a name="matrix"></a>

## Common matrix operations (`operator_C`, `operator_M`, and `operator_S`) <a name="matrix-operations"></a>

In [4]:
"""
The first step in defining a MatrixGroupElement is to initialize a matrix 
operation, which represents the group action.

Some common operations are (proper) rotations, reflections, and improper
rotations (rotoinversions). See examples below
"""

# Rotation around z-axis ([001]) by 2pi/3
ang = 2*np.pi/3
rotation = np.array([ [ np.cos(ang), -np.sin(ang), 0 ],
                      [ np.sin(ang),  np.cos(ang), 0 ],
                      [           0,            0, 1 ]])

print(f"Proper rotation around [001] axis by 2pi/3: \n {rotation}\n")

# Reflection through the xy-plane (normal vector = [001])
reflection = np.array([ [ 1,  0,  0 ], 
                        [ 0,  1,  0 ],
                        [ 0,  0, -1 ] ])

print(f"Reflection through [001] plane: \n {reflection}\n")

# Improper rotation around z-axis (same as a reflection through xy-plane 
# followed by a proper rotation) by 2pi/3
rotoinversion = rotation.dot(reflection)

print(f"Improper rotation around [001] axis by 2pi/3:\n {rotoinversion}\n")

Proper rotation around [001] axis by 2pi/3: 
 [[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]]

Reflection through [001] plane: 
 [[ 1  0  0]
 [ 0  1  0]
 [ 0  0 -1]]

Improper rotation around [001] axis by 2pi/3:
 [[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.    -1.   ]]



In [5]:
"""
Since these operations are so common, samosa provides functions 
operator_C, operator_M, operator_S that define proper rotations, reflections,
and rotoinversions, respectively. 
"""

# Same operators as in the last block, defined using built-in samosa functions

axis = [0,0,1]   # Axis of the rotation and the normal vector of the reflection
ang = 2*np.pi/3  # Angle of the rotation

print("Same matrix operations defined using samosa module:\n")

# Rotation around z-axis ([001]) by 2pi/3
rotation_m = operator_C(axis,ang)

print(f"Proper rotation around [001] axis by 2pi/3: \n {rotation_m}\n")

# Reflection through the xy-plane (normal vector = [001])
reflection_m = operator_M(axis)

print(f"Reflection through [001] plane: \n {reflection_m}\n")

# Improper rotation around z-axis (same as a reflection through xy-plane 
# followed by a proper rotation) by 2pi/3
rotoinversion_m = operator_S(axis,ang)

print(f"Improper rotation around [001] axis by 2pi/3:\n {rotoinversion_m}\n")

Same matrix operations defined using samosa module:

Proper rotation around [001] axis by 2pi/3: 
 [[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]]

Reflection through [001] plane: 
 [[ 1.  0. -0.]
 [-0.  1. -0.]
 [-0. -0. -1.]]

Improper rotation around [001] axis by 2pi/3:
 [[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.    -1.   ]]



In [6]:
"""
Since we often work with discrete rotations, operator_C and operator_S 
functions can also be defined using 

integer n, such that angle = 2*pi/n,
tuple of integers (k,n), such that angle = 2*pi*k/n
"""

# Shortcuts to defining discrete operations
np.all(operator_C(axis,3)==rotation), np.all(operator_C(axis,(1,3))==rotation)

(True, True)

In [7]:
# samosa will call a TypeError if the type of input is incorrect

#operator_C(axis, '3') # uncomment this to run operator_C with incorrect type
                      # for the angle argument (str)

In [8]:
# One must be cautious when defining the angles of rotations!

np.all(operator_C(axis,3)==operator_C(axis,3.0))

False

## Basis of the transformations <a name="matrix-basis"></a>

In [9]:
"""
In some cases, it is convenient to work with matrix operators in a basis that 
differs from Cartesian. Given column basis vectors B = [b1, b2, b3]
(for 3D space), a basis transformation of matrix operator O corresponds to

O_B = (B^-1) * O * B

Consider, for example a 3-fold rotation around [001] in the basis of the 
hexagonal lattice: 
"""

# Define basis vectors of the triangular lattice
B = np.array([ [    1,            0, 0 ],
               [ -0.5, np.sqrt(3)/2, 0 ],
               [    0,            0, 1 ]])

rotation_B = np.linalg.inv(B.T).dot(operator_C(axis,3)).dot(B.T)

print(f"Proper rotation around [001] axis by 2pi/3 "
      f"in hexagonal basis: \n {rotation_B}\n")


# Same result can be achieved by specifying basis argument in the operator_C
# (also applies to operator_M and operator_S)

rotation_B_m = operator_C(axis,3,basis=B)

print(f"Same as above, defined by specifying the `basis` argument "
      f"in operator_C: \n {rotation_B_m}\n")

Proper rotation around [001] axis by 2pi/3 in hexagonal basis: 
 [[ 0. -1.  0.]
 [ 1. -1.  0.]
 [ 0.  0.  1.]]

Same as above, defined by specifying the `basis` argument in operator_C: 
 [[ 0. -1.  0.]
 [ 1. -1.  0.]
 [ 0.  0.  1.]]



In [10]:
"""
The above information is contained in Help 
"""
help(operator_C)
help(operator_M)
help(operator_S)

Help on function operator_C in module samosa.symmetry.operations_3d:

operator_C(axis, angle, basis=None)
    Defines a proper 3D rotation.
    
    The general expression for a proper rotation around a 3D axis is found in
    (https://en.wikipedia.org/wiki/Rotation_matrix#Rotation_matrix_from_axis_and_angle)
    
    Arguments:
    axis  - np.1darray[3], rotation axis, not necesserally normalized;
    
    angle - if float, defines the angle of rotation;
            if int n, defines the angle of rotation as 2*pi/n;
            if tuple of ints (k, n), defines the angle of rotation as 2*pi*k/n;
    
    basis - None or np.2darray[3][3], (default=None), if not None,
            defines the basis of the transformation.
    
    Returns:
    rotation - np.2darray[3][3], 3D proper rotation matrix.

Help on function operator_M in module samosa.symmetry.operations_3d:

operator_M(axis, basis=None)
    Defines a 3D mirror reflection.
    
    Mirror reflection is defined as a 180 degree rota

## Initialization of the `MatrixGroupElement` object <a name="matrix-element-init"></a>

In [11]:
"""
MatrixGroupElement object is defined by specifying a matrix operator.
"""

# Define an element representing a 3-fold rotation
c3_matrix = operator_C([0,0,1],3)
c3_element = MatrixGroupElement(c3_matrix)

c3_element

MatrixGroupElement(operator=array([[-0.5  , -0.866,  0.   ],
       [ 0.866, -0.5  ,  0.   ],
       [ 0.   ,  0.   ,  1.   ]]), args_calculated=True)

## Product of two `MatrixGroupElement` objects <a name="matrix-element-prod"></a>

In [12]:
"""
Multiplication between two MatrixGroupElements A and B produces another
MatrixGroupElement with an operator defined as a matrix product AB.
"""

# Define an element representing a reflection
m_matrix = operator_M([0,0,1])
m_element = MatrixGroupElement(m_matrix)

# Multiply group elements
s3_element = c3_element * m_element

s3_element

MatrixGroupElement(operator=array([[-0.5  , -0.866,  0.   ],
       [ 0.866, -0.5  ,  0.   ],
       [ 0.   ,  0.   , -1.   ]]), args_calculated=True)

In [13]:
# We can verify that the product above is the same as simply defining S3
# element 

s3_element == MatrixGroupElement(operator_S([0,0,1],3))

True

## Inverse of a `MatrixGroupElement` object <a name="matrix-element-inv"></a>

In [14]:
"""
Inverse function returns a MatrixGroupElement defined by the matrix inverse of
the original operator.
"""

c3_inv_element = c3_element.inv
c3_inv_element

MatrixGroupElement(operator=array([[-0.5  ,  0.866,  0.   ],
       [-0.866, -0.5  ,  0.   ],
       [ 0.   ,  0.   ,  1.   ]]), args_calculated=True)

## Action of a `MatrixGroupElement` object on arrays <a name="matrix-element-act"></a>

In [15]:
"""
MatrixGroupElement can act on the left on arrays through matrix multiplication.
"""

# Define a 6-fold rotation
c6_element = MatrixGroupElement(operator_C([0,0,1],6))

c6_element * [1,0,0] # Can change vector type from list to np.array or tuple

array([0.5  , 0.866, 0.   ])

In [16]:
"""
Right multiplication is also defined.
"""

[1,0,0] * c6_element # Can change vector type from list to np.array or tuple

array([ 0.5  , -0.866,  0.   ])

## `MatrixGroupElement` object for identity matrix <a name="matrix-element-id"></a>

In [17]:
"""
We can check if the MatrixGroupElement is identity using .isidentity().
"""

c6_element.is_identity, MatrixGroupElement(np.eye(3)).is_identity

(False, True)

# `PermutationGroupElement` - permutation representation of group elements <a name="permutation"></a>

## Initialization of a `PermutationGroupElement` object <a name="permutation-element-init"></a>

In [18]:
"""
Another very useful representation of group elements is via permutations.
In samosa, PermutationGroupElement is used as a container for permutations.
"""

# To define a PermutationGroupElement, we need to provide an array of permuted
# integers (0-based)...

p_element_array = PermutationGroupElement([2,3,1,4,6,5])

print(f"Permutation defined using an array is:\n{p_element_array}\n")

# ...or a dictionary map from one integer set to another

p_element_dict = PermutationGroupElement({1 : 2,
                                          2 : 3,
                                          3 : 1,
                                          4 : 4,
                                          5 : 6,
                                          6 : 5})

print(f"Permutation defined using a dictionary is:\n{p_element_dict}")

Permutation defined using an array is:
(1, 2, 3)(5, 6)

Permutation defined using a dictionary is:
(1, 2, 3)(5, 6)


## Permutation of array elements and inegers <a name="permutation-element-act"></a>

In [19]:
"""
Permutations can act on arrays to permute their elements.
"""

p_element = PermutationGroupElement([2,3,1])

# Action on an array equal to the dimension of the permutation
array_1 = [23,42,121]
array_1_p = p_element * array_1

# Action on an array larger to the dimension of the permutation
array_2 = [59,194,295,3]
array_2_p = p_element * array_2

print(f"Action of a permutation {p_element} on an array {array_1} "
      f"yields \n{array_1_p}\n")

print(f"Action of a permutation {p_element} on an array {array_2} "
      f"yields \n{array_2_p}")

# Action on an array smaller to the dimension of the permutation produces 
# Exception

array_3 = [39,44]
#array_3_p = p_element * array_3 # Uncomment this to gen an Exception error

Action of a permutation (1, 2, 3) on an array [23, 42, 121] yields 
[42, 121, 23]

Action of a permutation (1, 2, 3) on an array [59, 194, 295, 3] yields 
[194, 295, 59, 3]


In [20]:
"""
Permutations are also defined for integers.
"""

int_1 = 0
int_2 = 1
int_3 = 3
int_4 = 5

print(f"Permutation {p_element} applied to {int_1} "
      f"gives {p_element * int_1}.\n")
print(f"Permutation {p_element} applied to {int_2} "
      f"gives {p_element * int_2}.\n")
print(f"Permutation {p_element} applied to {int_3} "
      f"gives {p_element * int_3}.\n")
print(f"Permutation {p_element} applied to {int_4} "
      f"gives {p_element * int_4}.\n")


Permutation (1, 2, 3) applied to 0 gives 0.

Permutation (1, 2, 3) applied to 1 gives 2.

Permutation (1, 2, 3) applied to 3 gives 1.

Permutation (1, 2, 3) applied to 5 gives 5.



## Product of two `PermutationGroupElement` objects <a name="permutation-element-prod"></a>

In [21]:
"""
Product of two permutations defines a new PermutationGroupElement with the same
dimension.
"""

p_12 = PermutationGroupElement([1,3,2])
p_01 = PermutationGroupElement([2,1,3])
p_prod = p_12*p_01

print(f"Checking the product of permutations:\n"
      f"{p_12} * {p_01} = {p_prod}")

Checking the product of permutations:
(2, 3) * (1, 2) = (1, 3, 2)


## Inverse of a `PermutationGroupElement` object <a name="permutation-element-inv"></a>

In [22]:
"""
As with the MatrixGroupElement, the inverse of a permutation is calculated
using .inv() method.
"""

p = PermutationGroupElement([2,3,1])
p_inv = p.inv

print(f"The inverse of permutation {p} is {p_inv}.\n"
      f"Their product is {p*p_inv}.")

The inverse of permutation (1, 2, 3) is (1, 3, 2).
Their product is ().


## `PermutationGroupElement` object for trivial permutation <a name="permutation-element-id"></a>

In [23]:
"""
We can check if the permutation is trivial (corresponds to the identity 
element).
"""

p.is_identity, (p * p_inv).is_identity

(False, True)

# `IdentityGroupElement` - generic identity group element <a name="identity"></a>

In [24]:
"""
IdentityGroupElement is a special class that defines an object that serves as a
'generic identity'.
"""

id_element = IdentityGroupElement()

# Action of IdentityGroupElement on any other object simply returns the back
# the object

var_list = [1, 
            np.array([1,2,432]), 
            "samosa",
            MatrixGroupElement(operator_C([0,0,1],5)),
            PermutationGroupElement([1,3,5,2,6,4])
           ]

for var in var_list:
    print(f"Action of identity on {var} gives {id_element * var}\n")


Action of identity on 1 gives 1

Action of identity on [  1   2 432] gives [  1   2 432]

Action of identity on samosa gives samosa

Action of identity on [[ 0.309 -0.951  0.   ]
 [ 0.951  0.309  0.   ]
 [ 0.     0.     1.   ]] gives [[ 0.309 -0.951  0.   ]
 [ 0.951  0.309  0.   ]
 [ 0.     0.     1.   ]]

Action of identity on (2, 3, 5, 6, 4) gives (2, 3, 5, 6, 4)



In [25]:
# Inverse of IdentityGroupElement is itself

id_element.inv

IdentityGroupElement(dim = None)

In [26]:
# We can initialize identity in matrix or permutation representation by passing
# Note that for this we must specify the dimension of the identity element

MatrixGroupElement(IdentityGroupElement(dim = 3)),\
PermutationGroupElement(IdentityGroupElement(dim = 3)) 

# Omission of dim argument yields Exception errors

#MatrixGroupElement(IdentityGroupElement())
#PermutationGroupElement(IdentityGroupElement())

(MatrixGroupElement(operator=array([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]]), args_calculated=True),
 PermutationGroupElement(permutation_cycles = [()]))